# 06 — Backtesting

Full backtest with transaction costs, performance metrics, and benchmark comparison.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

## 1. Generate Signals

In [ ]:
from src.data.price_fetcher import PriceFetcher
from src.strategy.signals import SignalGenerator

fetcher = PriceFetcher()
wti_df = fetcher.fetch_single('wti', start='2015-01-01')
prices = wti_df['Close']

# Simple MA crossover signals for demonstration
fast_ma = prices.rolling(20).mean()
slow_ma = prices.rolling(50).mean()
import pandas as pd
signals = pd.Series(
    ((fast_ma > slow_ma).astype(int) * 2 - 1),
    index=prices.index,
)
print(f"Signal distribution: {signals.value_counts().to_dict()}")

## 2. Run Backtest

In [ ]:
from src.backtesting.engine import BacktestConfig, BacktestEngine
from src.strategy.risk_manager import RiskManager

config = BacktestConfig(
    initial_capital=1_000_000,
    transaction_cost_bps=5,
    slippage_bps=2,
    start_date='2015-01-01',
)
engine = BacktestEngine(config=config)
result = engine.run(prices, signals)
print(f"Backtest complete: {len(result)} trading days")

## 3. Performance Analysis

In [ ]:
from src.backtesting.analysis import BacktestAnalysis

analysis = BacktestAnalysis(result)
analysis.print_summary()

## 4. Visualise Results

In [ ]:
from src.reporting.visualizations import Visualizer

viz = Visualizer()
fig_equity = viz.plot_equity_curve(result['portfolio_value'])
fig_equity.show()

fig_dd = viz.plot_drawdown(result['drawdown'])
fig_dd.show()